In [ ]:
!pip install -q \
    transformers==4.53.2 \
    peft==0.16.0 \
    trl==0.19.1 \
    bitsandbytes==0.46.1 \
    accelerate==1.8.1 \
    datasets==3.6.0 \
    evaluate==0.4.5 \
    sentencepiece

In [ ]:
# ============================================================
# Imports
# ============================================================

from pathlib import Path

import torch
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

In [ ]:
# ============================================================
# Fine-Tuning-PS-LLM
# Notebook: 01_baseline.ipynb
# Purpose: Configure and validate the execution environment.
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================================
# Project Paths
# ============================================================

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Fine-Tuning-PS-LLM")

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

PREDICTIONS_DIR = PROJECT_ROOT / "outputs" / "predictions"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"

PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ============================================================
# Load Dataset
# ============================================================

quotes = pd.read_csv(RAW_DATA_DIR / "quotes.csv")
gold = pd.read_csv(RAW_DATA_DIR / "gold_standard.csv")

dataset = quotes.merge(
    gold,
    on="quote_id",
    how="inner",
)

print(f"Samples : {len(dataset)}")

dataset.head()

Samples : 116


,quote_id,quote_text,category
0,Quote 01,"As an Agile Coach, how do I deal with argument...",Expressing concerns
1,Quote 02,How do I get my team to work in agile way and ...,Recommending changes
2,Quote 03,Do you think it is the team members responsibi...,Sharing negative feedback
3,Quote 04,I have made numerous suggestions (as the Scrum...,Expressing concerns
4,Quote 05,How can I deal with a developer who isn't on b...,Recommending changes


In [ ]:
dataset.head()

,quote_id,quote_text,category
0,Quote 01,"As an Agile Coach, how do I deal with argument...",Expressing concerns
1,Quote 02,How do I get my team to work in agile way and ...,Recommending changes
2,Quote 03,Do you think it is the team members responsibi...,Sharing negative feedback
3,Quote 04,I have made numerous suggestions (as the Scrum...,Expressing concerns
4,Quote 05,How can I deal with a developer who isn't on b...,Recommending changes


In [ ]:
# ============================================================
# Load Prompt Template
# ============================================================

PROMPTS_DIR = PROJECT_ROOT / "prompts"

with open(
    PROMPTS_DIR / "P01_Zero_Shot_Closed_Coding.txt",
    "r",
    encoding="utf-8"
) as f:
    P01_PROMPT = f.read()

with open(
    PROMPTS_DIR / "P02_Multi_Shot_Closed_Coding.txt",
    "r",
    encoding="utf-8"
) as f:
    P02_PROMPT = f.read()

print("P01 Prompt Loaded")
print("P02 Prompt Loaded")

P01 Prompt Loaded
P02 Prompt Loaded


In [ ]:
# ============================================================
# Load Qwen Model
# ============================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

print("Model ready.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model ready.


In [ ]:
# ============================================================
# Build Prompt for One Quote
# ============================================================

def build_prompt(prompt_template: str, quote_id: str, quote_text: str):

    return (
        prompt_template
        + "\n\n"
        + "Quotes:\n\n"
        + f"id_quote: {quote_id}\n"
        + f"related_quote: {quote_text}\n"
    )

In [ ]:
# ============================================================
# Predict One Quote
# ============================================================

import json

def predict_quote(model, tokenizer, prompt_template, quote_id, quote_text):

    prompt = (
        prompt_template
        + "\n\n"
        + "Quotes:\n\n"
        + f"id_quote: {quote_id}\n"
        + f"related_quote: {quote_text}\n"
    )

    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )

    return response

In [ ]:
quote = dataset.iloc[0]

response = predict_quote(
    model=model,
    tokenizer=tokenizer,
    prompt_template=P01_PROMPT,
    quote_id=quote["quote_id"],
    quote_text=quote["quote_text"],
)

print(response)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


```json
[
    {
        "id_quote": "Quote 01",
        "category": "Expressing concerns",
        "challenge_identified": "Team member is not a team player and is unwilling to work as a team, very arrogant and always tells the team that she is always right.",
        "related_quote": "As an Agile Coach, how do I deal with argumentative and uncooperative team members? [...] Team member is not a team player and is unwilling to work as a team, very arrogant and always tells the team that she is always right."
    }
]
```


In [ ]:

# ============================================================
# Parse Model Response
# ============================================================

import json

def parse_response(response: str):

    response = response.strip()

    if response.startswith("```json"):
        response = response.replace("```json", "").replace("```", "").strip()

    data = json.loads(response)

    return data[0]

In [ ]:
prediction = parse_response(response)

prediction

{'id_quote': 'Quote 01',
 'category': 'Expressing concerns',
 'challenge_identified': 'Team member is not a team player and is unwilling to work as a team, very arrogant and always tells the team that she is always right.',
 'related_quote': 'As an Agile Coach, how do I deal with argumentative and uncooperative team members? [...] Team member is not a team player and is unwilling to work as a team, very arrogant and always tells the team that she is always right.'}

In [ ]:
# ============================================================
# Import Inference Utilities
# ============================================================

import sys

sys.path.append(str(PROJECT_ROOT))

from utils.inference import (
    build_prompt,
    predict_quote,
    parse_response,
)

In [ ]:
from tqdm.auto import tqdm
import pandas as pd

# ==========================================================
# Configuration
# ==========================================================

ACTIVE_PROMPT = P02_PROMPT
PROMPT_NAME = "P02"

# None -> Run all quotes
# 5    -> Test with first 5 quotes
TEST_SIZE = None

# ==========================================================
# Output File
# ==========================================================

prediction_file = (
    PROJECT_ROOT
    / "outputs"
    / "predictions"
    / f"{PROMPT_NAME}_baseline_predictions.csv"
)

prediction_file.parent.mkdir(
    parents=True,
    exist_ok=True,
)

# ==========================================================
# Resume Previous Experiment
# ==========================================================

if prediction_file.exists():

    previous_results = pd.read_csv(prediction_file)

    results = previous_results.to_dict("records")

    processed_ids = set(previous_results["id_quote"])

    if TEST_SIZE is None:

        dataset_run = dataset[
            ~dataset["quote_id"].isin(processed_ids)
        ].copy()

    else:

        dataset_run = (
            dataset.head(TEST_SIZE)
            .loc[
                ~dataset.head(TEST_SIZE)["quote_id"].isin(processed_ids)
            ]
            .copy()
        )

    print("=" * 60)
    print("Resume Mode")
    print("=" * 60)
    print(f"Already processed : {len(results)}")
    print(f"Remaining quotes  : {len(dataset_run)}")
    print("=" * 60)

else:

    results = []

    if TEST_SIZE is None:
        dataset_run = dataset.copy()
    else:
        dataset_run = dataset.head(TEST_SIZE).copy()

    print("=" * 60)
    print("New Experiment")
    print("=" * 60)
    print(f"Quotes to process : {len(dataset_run)}")
    print("=" * 60)

# ==========================================================
# Run Inference
# ==========================================================

for _, row in tqdm(
    dataset_run.iterrows(),
    total=len(dataset_run),
    desc=f"{PROMPT_NAME} Baseline",
):

    try:

        response = predict_quote(
            model=model,
            tokenizer=tokenizer,
            prompt_template=ACTIVE_PROMPT,
            quote_id=row["quote_id"],
            quote_text=row["quote_text"],
        )

        prediction = parse_response(response)

        prediction["gold_category"] = row["category"]

        results.append(prediction)

        # Save after every prediction
        pd.DataFrame(results).to_csv(
            prediction_file,
            index=False,
        )

    except Exception as e:

        print(f"Skipped {row['quote_id']} -> {e}")

# ==========================================================
# Final DataFrame
# ==========================================================

results_df = pd.DataFrame(results)

print()
print("=" * 60)
print("Experiment Finished")
print("=" * 60)
print(f"Total predictions : {len(results_df)}")
print(f"Saved file        : {prediction_file}")
print("=" * 60)

results_df.head()

New Experiment
Quotes to process : 116


P02 Baseline:   0%|          | 0/116 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:447: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for mo

Skipped Quote 12 -> Expecting ',' delimiter: line 6 column 118 (char 306)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p'

Skipped Quote 44 -> Unterminated string starting at: line 6 column 26 (char 735)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p'


Experiment Finished
Total predictions : 114
Saved file        : /content/drive/MyDrive/Colab Notebooks/Fine-Tuning-PS-LLM/outputs/predictions/P02_baseline_predictions.csv


,id_quote,category,challenge_identified,related_quote,gold_category
0,Quote01,Seeking help,Difficulties obtaining support from a reluctan...,I have one expert engineer on our team but he ...,Expressing concerns
1,Quote02,Recommendations,Encouraging team to adopt agile vertical slici...,How do I get my team to work in agile way and ...,Recommending changes
2,Quote 03,Expressing concerns,Do you think it is the team members responsibi...,Do you think it is the team members responsibi...,Sharing negative feedback
3,Quote 04,Expressing concerns,Frustration with lack of acknowledgment of sug...,I have made numerous suggestions (as the Scrum...,Expressing concerns
4,Quote 05,Expressing concerns,Difficulties getting a developer who is resist...,But I just can't seem to get this person on bo...,Recommending changes


In [ ]:
results_df = pd.DataFrame(results)

results_df.head()

,id_quote,category,challenge_identified,related_quote,gold_category
0,Quote01,Seeking help,Difficulties obtaining support from a reluctan...,I have one expert engineer on our team but he ...,Expressing concerns
1,Quote02,Recommendations,Encouraging team to adopt agile vertical slici...,How do I get my team to work in agile way and ...,Recommending changes
2,Quote 03,Expressing concerns,Do you think it is the team members responsibi...,Do you think it is the team members responsibi...,Sharing negative feedback
3,Quote 04,Expressing concerns,Frustration with lack of acknowledgment of sug...,I have made numerous suggestions (as the Scrum...,Expressing concerns
4,Quote 05,Expressing concerns,Difficulties getting a developer who is resist...,But I just can't seem to get this person on bo...,Recommending changes


In [ ]:
import sys

sys.path.append(str(PROJECT_ROOT))

from utils.metrics import (
    evaluate_predictions,
    classification_results,
    compute_confusion_matrix,
    save_metrics,
)

In [ ]:
y_true = results_df["gold_category"]
y_pred = results_df["category"]

metrics = evaluate_predictions(
    y_true,
    y_pred,
)

print(metrics)

print()

print(classification_results(
    y_true,
    y_pred,
))

{'accuracy': 0.4473684210526316, 'cohen_kappa': np.float64(0.04570821153335103), 'macro_f1': 0.14996069182389937, 'weighted_f1': 0.29591470815403287}

                                    precision    recall  f1-score   support

                Admitting mistakes       0.00      0.00      0.00         3
Disagreeing with suggestions/ideas       0.00      0.00      0.00        33
       Drawing attention to errors       1.00      0.20      0.33         5
               Expressing concerns       0.45      0.98      0.62        50
                   Recommendations       0.00      0.00      0.00         0
              Recommending changes       0.00      0.00      0.00        14
                      Seeking help       0.33      0.20      0.25         5
         Sharing negative feedback       0.00      0.00      0.00         4

                          accuracy                           0.45       114
                         macro avg       0.22      0.17      0.15       114
           

In [ ]:
save_metrics(
    metrics,
    PROJECT_ROOT /
    "outputs/metrics/P01_baseline_metrics.json",
)

Metrics saved to /content/drive/MyDrive/Colab Notebooks/Fine-Tuning-PS-LLM/outputs/metrics/P01_baseline_metrics.json


In [ ]:
# Quick inference test

sample = dataset.iloc[0]

response = predict_quote(
    model=model,
    tokenizer=tokenizer,
    prompt_template=P01_PROMPT,
    quote_id=sample["quote_id"],
    quote_text=sample["quote_text"],
)

prediction = parse_response(response)

print(response)
print()
print(prediction)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:447: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


```json
[
    {
        "id_quote": "Quote 01",
        "category": "Expressing concerns",
        "challenge_identified": "Team member is not a team player and is unwilling to work as a team, very arrogant and always tells the team that she is always right.",
        "related_quote": "As an Agile Coach, how do I deal with argumentative and uncooperative team members? [...] Team member is not a team player and is unwilling to work as a team, very arrogant and always tells the team that she is always right."
    }
]
```

{'id_quote': 'Quote 01', 'category': 'Expressing concerns', 'challenge_identified': 'Team member is not a team player and is unwilling to work as a team, very arrogant and always tells the team that she is always right.', 'related_quote': 'As an Agile Coach, how do I deal with argumentative and uncooperative team members? [...] Team member is not a team player and is unwilling to work as a team, very arrogant and always tells the team that she is always right.'}
